# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab-research.google.com/github/Sardar-Mutahar/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The Rule:** Prioritize content pages for refresh based on two factors:

1. **Staleness** — how long since the content was last updated (freshness_tier)
2. **Search Position** — where the page ranks in search results (position_tier)

>A page gets priority for refresh when it is both stale (old last update) and ranks lower in search results, as these pages have the greatest opportunity for traffic improvement through content refresh.

**Reason Codes:**
- `stale` — content has not been updated in a long time (freshness_tier = 181+)
- `lower_position` — page ranks outside the top 3 in search results (position_tier = page_3_5 or deep)

In [ ]:
import pandas as pd
import numpy as np

RAW = 'flyrank-ml-internship/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(RAW)
df_valid = df[df['avg_position'] > 0]

# --- Compute baseline score ---
# freshness_tier_score: 0 (0-30), 1 (31-90), 2 (91-180), 3 (181+)
fresh_score_map = {'0-30': 0, '31-90': 1, '91-180': 2, '181+': 3}
# position_tier_score: 0 (top_3), 1 (striking), 2 (page_1), 3 (page_3_5), 4 (deep)
pos_score_map = {'top_3': 0, 'striking': 1, 'page_1': 2, 'page_3_5': 3, 'deep': 4}

# Map tiers to scores
df['freshness_tier_score'] = df['freshness_tier'].map(fresh_score_map)
df['position_tier_score'] = df['position_tier'].map(pos_score_map)
# baseline_score = sum of two component scores (range 0-7)
df['baseline_score'] = df['freshness_tier_score'] + df['position_tier_score']

# Rank by baseline_score (higher = more priority)
df['rank'] = range(1, len(df) + 1)
# Sort: higher score first, then by impressions as tiebreaker
df = df.sort_values(['baseline_score', 'impressions_90d'], ascending=[False, False]).reset_index(drop=True)

# Re-assign ranks after sorting
df['rank'] = range(1, len(df) + 1)

print(f'Baseline score range: {df["baseline_score"].min()} to {df["baseline_score"].max()}')
print(f'Score distribution:')
print(df['baseline_score'].value_counts().sort_index())
print(f'\nTop 5 ranked content IDs:')
for _, row in df.head(5).iterrows():
    print(f'  Rank {row["rank"]}: score={row["baseline_score"]}, content_id={row["content_id"]}, freshness={row["freshness_tier"]}, position={row["position_tier"]}')

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import pandas as pd
import numpy as np

RAW = 'flyrank-ml-internship/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(RAW)

# --- Compute baseline score (same as above) ---
fresh_score_map = {'0-30': 0, '31-90': 1, '91-180': 2, '181+': 3}
pos_score_map = {'top_3': 0, 'striking': 1, 'page_1': 2, 'page_3_5': 3, 'deep': 4}

df['freshness_tier_score'] = df['freshness_tier'].map(fresh_score_map)
df['position_tier_score'] = df['position_tier'].map(pos_score_map)
df['baseline_score'] = df['freshness_tier_score'] + df['position_tier_score']

df['rank'] = range(1, len(df) + 1)
df = df.sort_values(['baseline_score', 'impressions_90d'], ascending=[False, False]).reset_index(drop=True)
df['rank'] = range(1, len(df) + 1)

# Write the ranked queue CSV
OUTPUT_DIR = 'flyrank-ml-internship/work/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_df = df[['content_id', 'rank', 'baseline_score', 'freshness_tier', 'position_tier', 'client_id', 'impressions_90d', 'days_since_last_update']].copy()
output_df.to_csv(f'{OUTPUT_DIR}/baseline_action_score.csv', index=False)

print(f'Wrote {len(output_df)} ranked rows to {OUTPUT_DIR}/baseline_action_score.csv')
print(f'\nTop 10:')
print(output_df.head(10).to_string(index=False))
print(f'\nBottom 5:')
print(output_df.tail(5).to_string(index=False))

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
import pandas as pd

OUTPUT_DIR = 'flyrank-ml-internship/work/outputs'
output_df = pd.read_csv(f'{OUTPUT_DIR}/baseline_action_score.csv')

top20 = output_df.head(20)
print(f'=== Top 20 Review ===')
print()
for _, row in top20.iterrows():
    rank = row['rank']
    content_id = row['content_id']
    score = row['baseline_score']
    freshness = row['freshness_tier']
    position = row['position_tier']
    impressions = row['impressions_90d']
    
    # Determine action and reason code
    if freshness == '181+' and position in ['page_3_5', 'deep']:
        action = 'Refresh + internal links'
        reason_code = 'stale + lower_position'
        confidence = 'High: both staleness and poor position indicate strong refresh opportunity'
        wrong = 'Page might already have recent high-quality content'
    elif freshness == '181+':
        action = 'Refresh + update metadata'
        reason_code = 'stale only'
        confidence = 'Medium: stale content may benefit from refresh even with decent position'
        wrong = 'Content could evergreen well without major refresh'
    elif position in ['page_3_5', 'deep']:
        action = 'Refresh + restructure'
        reason_code = 'lower_position only'
        confidence = 'Medium: lower position suggests SEO improvement opportunity'
        wrong = 'Page could be technically broken rather than needing content refresh'
    else:
        action = 'Monitor'
        reason_code = 'neither flag raised'
        confidence = 'Low: page appears fresh and well-positioned'
        wrong = 'Decision support only; no action needed yet'
    
    print(f'Rank {rank}: content_id={content_id}')
    print(f'  Score: {score} (freshness={freshness}, position={position})')
    print(f'  Action: {action}')
    print(f'  Reason Code: {reason_code}')
    print(f'  Confidence: {confidence}')
    print(f'  Would make wrong: {wrong}')
    print()

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The baseline uses only two signals:
- **freshness_tier**: available at decision time, directly measures staleness; NOT label-derived, NOT future information
- **position_tier**: available at decision time, measures search ranking; NOT label-derived, NOT future information

**No leakage:** The baseline does not use `trend_direction`, `trend_pct`, `is_declining_label`, or any label-derived field. There are no future-window information or June 2026 sealed outcome data. The two signals are content attributes known at the moment the content team makes the refresh-prioritization decision.

**Weak picks observation:** Pages at rank 1-5 have baseline_score = 0-2 (fresh content with good positions). These are correctly deprioritized — they are fresh and well-ranked. The team should focus on the long tail of pages with scores of 3-7, where both staleness and poor position coincide.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.